# Context-Aware RL Crop Stress and Recommendation System — Option A (Revised v3 — pH sensitivity audit)

**Purpose:** implement the manuscript's hybrid architecture with genuine tabular Q-learning for *pseudo-temporal stress progression* and a separate rule-constrained agronomic recommendation layer.

> **Important:** The source CSV has no timestamp or repeated field identifier. Therefore, transitions constructed by nearest-neighbour matching are *pseudo-temporal similarity transitions*, not observed chronological transitions. Do not describe their test MAE as validated future-forecast accuracy. The CSV also has no nitrogen or potassium concentration columns, so those Table 4 rules cannot be evaluated unless those measurements are supplied.

Run the cells in this revised notebook from top to bottom. The original notebook is retained separately and is not overwritten.

## Step 1 — Imports, data loading and schema audit

The uploaded file is `agriculture_dataset(4).csv`. In Colab, upload it to the runtime or change `DATA_PATH`. This section deliberately checks that the required CSI input columns exist.

In [ ]:
import os, json, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score

RANDOM_STATE = 42
DATA_PATH = "/mnt/data/agriculture_dataset(4).csv"  # Change to "agriculture_dataset(4).csv" in Colab if uploaded there
if not Path(DATA_PATH).exists() and Path("/content/agriculture_dataset(4).csv").exists():
    DATA_PATH = "/content/agriculture_dataset(4).csv"

if not Path(DATA_PATH).exists():
    raise FileNotFoundError(f"Cannot find {DATA_PATH}. Upload the CSV and update DATA_PATH.")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

REQUIRED = ["Soil_Moisture", "Chlorophyll_Content", "Soil_pH", "NDVI", "Temperature", "Humidity", "Rainfall", "Crop_Health_Label"]
missing_required = [c for c in REQUIRED if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing from CSV: {missing_required}")

print("\nCrop distribution:")
print(df["Crop_Type"].value_counts(dropna=False) if "Crop_Type" in df else "Crop_Type not available")
print("\nGrowth-stage distribution:")
print(df["Crop_Growth_Stage"].value_counts(dropna=False) if "Crop_Growth_Stage" in df else "Crop_Growth_Stage not available")
print("\nAvailable N/K columns:", [c for c in df.columns if any(k in c.lower() for k in ["nitrogen", "potassium"])])

## Step 2 — Freeze the CSI component definitions before generating labels

The manuscript's agreed composite is `CSI = 0.50 M + 0.30 C + 0.20 P`. This notebook does **not** silently guess the component transformations. The current CSV contains measured numeric columns but does not define their stress orientation or the crop-/soil-specific optimum pH.

Before running this cell, confirm:
1. Lower soil moisture means greater moisture stress (`low_is_stress`).
2. Lower chlorophyll means greater chlorophyll stress (`low_is_stress`).
3. The pH method is not documented by the dataset provider. The crop-specific intervals below are therefore provisional literature references and must be sensitivity-tested.

Normalisation minima/maxima and the five CSI quintile cut-points are estimated from the training split only, then frozen for validation/test. The binary `Crop_Health_Label` is used only for stratification/audit, not as the five-class target.

## Crop-specific soil-pH reference intervals

The pH-stress component uses the distance outside a crop-specific optimum interval rather than distance from one universal pH point. The intervals used here are Wheat 6.0–7.5, Maize 6.0–7.5, and Rice 4.0–6.0 (soil pH measured in water), as tabulated in FAO Soils Bulletin No. 56, Table 13. The deviation is zero when a sample lies within its crop interval and increases with distance outside it. The maximum deviation observed in the training partition is used for scaling; validation and test data do not refit this scale.

**Important dataset limitation:** the Kaggle CSV does not specify the soil-pH measurement method or soil:water ratio. Therefore, the FAO intervals are used as literature-based reference values for operationalising the index, not as verified calibration thresholds for the dataset. This limitation must be disclosed in the manuscript and should be checked against the dataset documentation or original data provider.

**Source:** FAO, *Soil management: compost production and use in tropical and subtropical environments*, Soils Bulletin No. 56, Table 13 (optimum soil reaction ranges of crops): https://www.fao.org/4/s8930e/s8930e.pdf


In [ ]:
# ---------------- AGRONOMIC CONFIGURATION ----------------
CSI_WEIGHTS = {"M": 0.50, "C": 0.30, "P": 0.20}
MOISTURE_DIRECTION = "low_is_stress"
CHLOROPHYLL_DIRECTION = "low_is_stress"

# Crop-specific optimum soil-pH intervals (pH measured in water),
# based on FAO Soils Bulletin No. 56, Table 13.
# These are reference intervals, not measured crop-specific optima in this dataset.
PH_OPTIMUM_RANGES = {
    "Wheat": (6.0, 7.5),
    "Maize": (6.0, 7.5),
    "Rice":  (4.0, 6.0),
}

# Optional threshold used only as an additional independent constraint for
# the severe multi-stress crop-switch rule. Keep None unless independently justified.
PH_DEVIATION_THRESHOLD = None

if MOISTURE_DIRECTION not in {"low_is_stress", "high_is_stress"}:
    raise ValueError("Invalid MOISTURE_DIRECTION")
if CHLOROPHYLL_DIRECTION not in {"low_is_stress", "high_is_stress"}:
    raise ValueError("Invalid CHLOROPHYLL_DIRECTION")
assert np.isclose(sum(CSI_WEIGHTS.values()), 1.0), "CSI weights must sum to 1.0"
if "Crop_Type" not in df.columns:
    raise ValueError("Crop_Type is required for crop-specific pH reference intervals.")
unknown_crops = sorted(set(df["Crop_Type"].dropna().astype(str)) - set(PH_OPTIMUM_RANGES))
if unknown_crops:
    raise ValueError(f"No pH reference interval defined for crop(s): {unknown_crops}")

# Numeric conversion and complete-case filtering for the inputs used in this pipeline.
work = df.copy()
for col in REQUIRED:
    work[col] = pd.to_numeric(work[col], errors="coerce")
work["Crop_Type"] = work["Crop_Type"].astype(str)
work = work.dropna(subset=REQUIRED + ["Crop_Type"]).reset_index(drop=True)

# 70/15/15 split. Binary health label is used only to preserve its distribution.
train_df, temp_df = train_test_split(
    work, test_size=0.30, random_state=RANDOM_STATE,
    stratify=work["Crop_Health_Label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_STATE,
    stratify=temp_df["Crop_Health_Label"]
)
train_df = train_df.copy().reset_index(drop=True)
val_df = val_df.copy().reset_index(drop=True)
test_df = test_df.copy().reset_index(drop=True)

# Training-only scaling parameters for moisture, chlorophyll and pH deviation.
def fit_minmax(series):
    return float(series.min()), float(series.max())

def crop_ph_deviation(frame):
    """Distance outside the crop-specific optimum pH interval; zero inside it."""
    bounds = frame["Crop_Type"].map(PH_OPTIMUM_RANGES)
    lower = bounds.map(lambda x: x[0]).astype(float)
    upper = bounds.map(lambda x: x[1]).astype(float)
    return np.maximum(lower.to_numpy() - frame["Soil_pH"].to_numpy(), 0.0) + np.maximum(frame["Soil_pH"].to_numpy() - upper.to_numpy(), 0.0)

m_min, m_max = fit_minmax(train_df["Soil_Moisture"])
c_min, c_max = fit_minmax(train_df["Chlorophyll_Content"])
p_train_dev = crop_ph_deviation(train_df)
p_scale = float(np.max(p_train_dev))
if np.isclose(m_max, m_min) or np.isclose(c_max, c_min) or np.isclose(p_scale, 0):
    raise ValueError("A CSI component has zero training range; inspect the data/configuration.")

CSI_PARAMS = {
    "soil_moisture_min": m_min, "soil_moisture_max": m_max,
    "chlorophyll_min": c_min, "chlorophyll_max": c_max,
    "crop_specific_ph_optimum_ranges": PH_OPTIMUM_RANGES,
    "ph_deviation_scale_train": p_scale,
    "weights": CSI_WEIGHTS.copy(),
}

def add_csi_components(frame):
    out = frame.copy()
    m_norm = ((out["Soil_Moisture"] - m_min) / (m_max - m_min)).clip(0, 1)
    c_norm = ((out["Chlorophyll_Content"] - c_min) / (c_max - c_min)).clip(0, 1)
    if MOISTURE_DIRECTION == "low_is_stress":
        out["M_stress"] = 1 - m_norm
    else:
        out["M_stress"] = m_norm
    if CHLOROPHYLL_DIRECTION == "low_is_stress":
        out["C_stress"] = 1 - c_norm
    else:
        out["C_stress"] = c_norm
    out["PH_deviation"] = crop_ph_deviation(out)
    out["P_stress"] = (out["PH_deviation"] / p_scale).clip(0, 1)
    out["CSI"] = (CSI_WEIGHTS["M"] * out["M_stress"] + CSI_WEIGHTS["C"] * out["C_stress"] + CSI_WEIGHTS["P"] * out["P_stress"])
    return out

train_df = add_csi_components(train_df)
val_df = add_csi_components(val_df)
test_df = add_csi_components(test_df)

# Quintile cut-points learned on TRAIN only; frozen for validation and test.
CSI_CUTS = np.quantile(train_df["CSI"], [0.20, 0.40, 0.60, 0.80]).astype(float)
def assign_stress_class(frame):
    out = frame.copy()
    out["Stress_Level"] = np.searchsorted(CSI_CUTS, out["CSI"].to_numpy(), side="right").astype(int)
    return out

train_df = assign_stress_class(train_df)
val_df = assign_stress_class(val_df)
test_df = assign_stress_class(test_df)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Crop-specific pH reference intervals:", PH_OPTIMUM_RANGES)
print("Training CSI cut-points:", CSI_CUTS)
print("Training stress distribution:")
print(train_df["Stress_Level"].value_counts().sort_index())
print("Validation stress distribution:")
print(val_df["Stress_Level"].value_counts().sort_index())
print("Test stress distribution:")
print(test_df["Stress_Level"].value_counts().sort_index())
print("\nCSI parameters (save these for reproducibility):")
print(json.dumps(CSI_PARAMS, indent=2))


## Step 2A — pH sensitivity and measurement-scale audit

The dataset description does not state the pH measurement medium. This cell therefore compares (i) the provisional crop-specific literature intervals, (ii) a common 6.0–7.5 analytical comparator used only to test sensitivity (not asserted as a universal crop optimum), and (iii) an index without pH. For each configuration, moisture/chlorophyll/pH scaling and CSI quintile cut-points are fitted on the training partition only and then applied unchanged to validation/test. The pH-free ablation renormalises M and C weights to 0.625 and 0.375. These comparisons assess label sensitivity; they do not validate physiological severity.

The unit and interpretation of `Soil_Moisture` and `Chlorophyll_Content` are also not established by the CSV column descriptions. Their numeric ranges are printed for audit, but no unit is inferred.

In [ ]:
# Numeric-scale audit: ranges are descriptive, not proof of units.
for col in ["Soil_Moisture", "Chlorophyll_Content", "Soil_pH"]:
    print(f"{col}: min={work[col].min():.6g}, max={work[col].max():.6g}, mean={work[col].mean():.6g}, std={work[col].std():.6g}")

COMMON_PH_COMPARATOR = (6.0, 7.5)  # analytical comparator only; not a universal agronomic optimum

def fit_config_and_label(train_part, eval_parts, ph_mode):
    # All scalers and quintile thresholds are learned from training data only.
    mlo, mhi = train_part["Soil_Moisture"].min(), train_part["Soil_Moisture"].max()
    clo, chi = train_part["Chlorophyll_Content"].min(), train_part["Chlorophyll_Content"].max()
    def deviation(frame):
        if ph_mode == "no_pH":
            return np.zeros(len(frame), dtype=float)
        if ph_mode == "crop_specific_FAO_provisional":
            b = frame["Crop_Type"].map(PH_OPTIMUM_RANGES)
            lo = b.map(lambda x: x[0]).to_numpy(float); hi = b.map(lambda x: x[1]).to_numpy(float)
        elif ph_mode == "common_comparator_6_0_7_5":
            lo = np.full(len(frame), COMMON_PH_COMPARATOR[0]); hi = np.full(len(frame), COMMON_PH_COMPARATOR[1])
        else:
            raise ValueError(ph_mode)
        ph = frame["Soil_pH"].to_numpy(float)
        return np.maximum(lo-ph, 0) + np.maximum(ph-hi, 0)
    p_train = deviation(train_part)
    pscale = float(p_train.max()) if ph_mode != "no_pH" else 1.0
    if np.isclose(mhi,mlo) or np.isclose(chi,clo) or (ph_mode != "no_pH" and np.isclose(pscale,0)):
        raise ValueError("Zero-range component in sensitivity configuration")
    def score(frame):
        m = ((frame["Soil_Moisture"]-mlo)/(mhi-mlo)).clip(0,1).to_numpy()
        c = ((frame["Chlorophyll_Content"]-clo)/(chi-clo)).clip(0,1).to_numpy()
        M = 1-m if MOISTURE_DIRECTION == "low_is_stress" else m
        C = 1-c if CHLOROPHYLL_DIRECTION == "low_is_stress" else c
        P = np.clip(deviation(frame)/pscale,0,1) if ph_mode != "no_pH" else np.zeros(len(frame))
        if ph_mode == "no_pH":
            return 0.625*M + 0.375*C
        return 0.50*M + 0.30*C + 0.20*P
    tr_score = score(train_part)
    cuts = np.quantile(tr_score,[.2,.4,.6,.8])
    result = {"train":train_part.copy()}
    for name, frame in eval_parts.items():
        z=frame.copy(); z["CSI_sensitivity"]=score(frame)
        z["Stress_sensitivity"]=np.searchsorted(cuts,z["CSI_sensitivity"].to_numpy(),side="right")
        result[name]=z
    return cuts,result

SENSITIVITY_MODES = ["crop_specific_FAO_provisional", "common_comparator_6_0_7_5", "no_pH"]
sensitivity_outputs = {}
sensitivity_summary=[]
for mode in SENSITIVITY_MODES:
    cuts, parts = fit_config_and_label(train_df, {"validation":val_df,"test":test_df}, mode)
    sensitivity_outputs[mode]=parts
    for split_name, part in parts.items():
        counts=part["Stress_sensitivity"].value_counts().reindex(range(5),fill_value=0)
        row={"configuration":mode,"split":split_name,"n":len(part),"cut_20":cuts[0],"cut_40":cuts[1],"cut_60":cuts[2],"cut_80":cuts[3]}
        row.update({f"class_{k}_count":int(v) for k,v in counts.items()})
        row.update({f"class_{k}_pct":float(100*v/len(part)) for k,v in counts.items()})
        sensitivity_summary.append(row)

sensitivity_summary_df=pd.DataFrame(sensitivity_summary)
print("Stress-class distribution sensitivity:")
display(sensitivity_summary_df.round(3))

# Compare validation/test class changes against the primary crop-specific configuration.
base=sensitivity_outputs["crop_specific_FAO_provisional"]
change_rows=[]
for mode in SENSITIVITY_MODES[1:]:
    for split_name in ["validation","test"]:
        a=base[split_name]["Stress_sensitivity"].to_numpy()
        b=sensitivity_outputs[mode][split_name]["Stress_sensitivity"].to_numpy()
        change_rows.append({"comparison":f"{mode} vs crop-specific provisional", "split":split_name,
                            "changed_samples":int(np.sum(a!=b)),"total_samples":len(a),
                            "changed_percent":100*float(np.mean(a!=b)),
                            "spearman_rank_correlation":float(pd.Series(base[split_name]["CSI_sensitivity"]).corr(
                                pd.Series(sensitivity_outputs[mode][split_name]["CSI_sensitivity"]),method="spearman"))})
change_df=pd.DataFrame(change_rows)
print("\nClass-change and rank-correlation analysis:")
display(change_df.round(4))

# Crop-wise distribution for the primary provisional pH configuration.
print("\nTest-set class distribution by crop (crop-specific provisional pH):")
display(pd.crosstab(base["test"]["Crop_Type"],base["test"]["Stress_sensitivity"],normalize="index").round(3))


## Step 3 — Construct RL states and training-only pseudo-transitions

**State:** five-level CSI stress class plus discretised NDVI, soil moisture, pH, temperature, humidity and rainfall.

**RL action:** predicted stress increment `Δs ∈ {0, 1, 2}` (as described in the current manuscript). The resulting estimate is clipped to `[0, 4]`.

**Pseudo-transition construction:** for each source observation, find a nearest *different* observation from the training partition with stress class equal to the current class or up to two levels higher. This creates a similarity-based transition, not a time-ordered observation. Neighbour search is fitted on training data only. If no eligible neighbour is found, a self-transition is used and flagged.

This construction must be described transparently in the paper. It cannot establish real temporal forecasting.

In [ ]:
STATE_FEATURES = ["NDVI", "Soil_Moisture", "Soil_pH", "Temperature", "Humidity", "Rainfall"]
missing_state = [c for c in STATE_FEATURES if c not in train_df.columns]
if missing_state:
    raise ValueError(f"Missing state features: {missing_state}")

# Fit scaling and discretisation edges on training observations only.
state_scaler = StandardScaler().fit(train_df[STATE_FEATURES])
Xtr_scaled = state_scaler.transform(train_df[STATE_FEATURES])

STATE_BINS = {}
for j, col in enumerate(STATE_FEATURES):
    qs = np.quantile(Xtr_scaled[:, j], [1/3, 2/3])
    STATE_BINS[col] = np.unique(qs)

def encode_states(frame):
    scaled = state_scaler.transform(frame[STATE_FEATURES])
    encoded = []
    for i, (_, row) in enumerate(frame.iterrows()):
        bins = tuple(int(np.searchsorted(STATE_BINS[col], scaled[i, j], side="right")) for j, col in enumerate(STATE_FEATURES))
        encoded.append((int(row["Stress_Level"]),) + bins)
    return encoded

train_states = encode_states(train_df)
val_states = encode_states(val_df)
test_states = encode_states(test_df)

# Fit nearest-neighbour search on training rows only.
K_NEIGHBOURS = 20
nn_model = NearestNeighbors(n_neighbors=min(K_NEIGHBOURS + 1, len(train_df)), metric="euclidean", n_jobs=-1)
nn_model.fit(Xtr_scaled)

MAX_TRANSITIONS = 20000  # reproducible computational cap; increase if resources permit
rng = np.random.default_rng(RANDOM_STATE)
if len(train_df) > MAX_TRANSITIONS:
    train_source_idx = np.sort(rng.choice(len(train_df), MAX_TRANSITIONS, replace=False))
else:
    train_source_idx = np.arange(len(train_df))

def make_pseudo_transitions(source_frame, source_states, source_scaled, source_indices=None):
    distances, neighbours = nn_model.kneighbors(source_scaled, return_distance=True)
    transitions = []
    no_valid_neighbour = 0
    for local_i, (dist_row, nbr_row) in enumerate(zip(distances, neighbours)):
        source_stress = int(source_frame.iloc[local_i]["Stress_Level"])
        chosen = None
        for j in nbr_row:
            # Exclude exact same training row when source is part of training.
            if source_indices is not None and int(j) == int(source_indices[local_i]):
                continue
            candidate_stress = int(train_df.iloc[int(j)]["Stress_Level"])
            if source_stress <= candidate_stress <= min(4, source_stress + 2):
                chosen = int(j)
                break
        if chosen is None:
            no_valid_neighbour += 1
            chosen = int(nbr_row[0])
            # If no admissible upward/stable neighbour exists, force a self-transition.
            target_stress = source_stress
            next_state = source_states[local_i]
        else:
            target_stress = int(train_df.iloc[chosen]["Stress_Level"])
            next_state = train_states[chosen]
        transitions.append((source_states[local_i], source_stress, target_stress, next_state))
    return transitions, no_valid_neighbour

train_source = train_df.iloc[train_source_idx].reset_index(drop=True)
train_source_states = [train_states[i] for i in train_source_idx]
train_source_scaled = Xtr_scaled[train_source_idx]
train_transitions, train_no_neighbour = make_pseudo_transitions(
    train_source, train_source_states, train_source_scaled, train_source_idx
)
print("Training pseudo-transitions:", len(train_transitions))
print("Transitions without admissible neighbour (self-transition fallback):", train_no_neighbour)
print("Training transitions with self-transition fallback (%):", 100*train_no_neighbour/len(train_transitions))

## Step 4 — Genuine tabular Q-learning for stress-increment selection

The agent uses ε-greedy exploration and the Q-learning update:

`Q(s,a) ← Q(s,a) + α [r + γ maxₐ′ Q(s′,a′) − Q(s,a)]`

The reward is the negative absolute error between the stress level implied by the selected increment and the pseudo-next stress class. Parameters initially follow the manuscript: `α=0.1`, `γ=0.9`, `ε=0.2`, 100 episodes. These are implementation parameters, not evidence of optimality. The next state is the matched training neighbour's encoded state, so the discount term can contribute to the update.

In [ ]:
ACTIONS = np.array([0, 1, 2], dtype=int)  # predicted stress increment
ALPHA = 0.10
GAMMA = 0.90
EPSILON = 0.20
EPISODES = 100


def train_q_learning(transitions, seed=RANDOM_STATE):
    local_rng = np.random.default_rng(seed)
    Q = defaultdict(lambda: np.zeros(len(ACTIONS), dtype=float))
    episode_returns = []
    episode_td_mse = []
    policy_history = []

    for episode in range(EPISODES):
        order = local_rng.permutation(len(transitions))
        total_reward = 0.0
        td_errors = []
        for idx in order:
            state, current_stress, target_next_stress, next_state = transitions[int(idx)]
            if local_rng.random() < EPSILON:
                action_idx = int(local_rng.integers(len(ACTIONS)))
            else:
                action_idx = int(np.argmax(Q[state]))
            delta = int(ACTIONS[action_idx])
            predicted_next = min(4, current_stress + delta)
            reward = -float(abs(predicted_next - target_next_stress))
            td_target = reward + GAMMA * float(np.max(Q[next_state]))
            td_error = td_target - Q[state][action_idx]
            Q[state][action_idx] += ALPHA * td_error
            total_reward += reward
            td_errors.append(td_error ** 2)

        episode_returns.append(total_reward)
        episode_td_mse.append(float(np.mean(td_errors)) if td_errors else np.nan)
        policy_history.append({s: int(np.argmax(v)) for s, v in Q.items()})

    return Q, np.asarray(episode_returns), np.asarray(episode_td_mse), policy_history

Q_table, episode_returns, td_mse, policy_history = train_q_learning(train_transitions)

plt.figure(figsize=(8, 4))
plt.plot(episode_returns, alpha=0.35, label="Episode return")
window = min(10, len(episode_returns))
if window > 1:
    smooth = pd.Series(episode_returns).rolling(window, min_periods=1).mean()
    plt.plot(smooth, label=f"{window}-episode moving average")
plt.xlabel("Episode")
plt.ylabel("Cumulative reward")
plt.title("Q-learning training return")
plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.plot(td_mse)
plt.xlabel("Episode")
plt.ylabel("Mean squared TD error")
plt.title("Temporal-difference error during Q-learning")
plt.tight_layout(); plt.show()

print("Q-table states learned:", len(Q_table))
print("Final 10-episode mean return:", float(np.mean(episode_returns[-10:])))

## Step 5 — Evaluate pseudo-transition predictions on validation and test sets

For validation/test observations, the nearest-neighbour reference is restricted to the training partition. The target remains a *pseudo-next stress class* supplied by the matched training observation. Report the result as pseudo-transition agreement/error, not as observed future-stress forecasting accuracy.

Persistence (`Δs=0`) is included as a transparent baseline. No test observation is used to fit the scaler, state bins, neighbour index or Q-table.

In [ ]:
def make_eval_transitions(eval_frame, eval_states, eval_scaled):
    distances, neighbours = nn_model.kneighbors(eval_scaled, return_distance=True)
    out = []
    for i, nbr_row in enumerate(neighbours):
        s = int(eval_frame.iloc[i]["Stress_Level"])
        chosen = None
        for j in nbr_row:
            ns = int(train_df.iloc[int(j)]["Stress_Level"])
            if s <= ns <= min(4, s + 2):
                chosen = int(j)
                break
        if chosen is None:
            target = s
            next_state = eval_states[i]
        else:
            target = int(train_df.iloc[chosen]["Stress_Level"])
            next_state = train_states[chosen]
        out.append((eval_states[i], s, target, next_state))
    return out

val_scaled = state_scaler.transform(val_df[STATE_FEATURES])
test_scaled = state_scaler.transform(test_df[STATE_FEATURES])
val_transitions = make_eval_transitions(val_df, val_states, val_scaled)
test_transitions = make_eval_transitions(test_df, test_states, test_scaled)

def predict_increment(Q, state):
    return int(ACTIONS[int(np.argmax(Q[state]))])

def evaluate_pseudo_transitions(transitions, Q, split_name):
    y_true, y_pred, y_persist = [], [], []
    for state, current, target, _ in transitions:
        delta = predict_increment(Q, state)
        y_true.append(int(target))
        y_pred.append(min(4, int(current) + delta))
        y_persist.append(int(current))
    result = {
        "split": split_name,
        "n": len(y_true),
        "Q_learning_pseudo_transition_MAE": mean_absolute_error(y_true, y_pred),
        "persistence_pseudo_transition_MAE": mean_absolute_error(y_true, y_persist),
        "Q_learning_exact_agreement": accuracy_score(y_true, y_pred),
        "Q_learning_within_1_level": float(np.mean(np.abs(np.asarray(y_true)-np.asarray(y_pred)) <= 1)),
    }
    return result, np.asarray(y_true), np.asarray(y_pred)

val_metrics, yv, pv = evaluate_pseudo_transitions(val_transitions, Q_table, "validation")
test_metrics, yt, pt = evaluate_pseudo_transitions(test_transitions, Q_table, "test")
metrics_df = pd.DataFrame([val_metrics, test_metrics])
print(metrics_df.to_string(index=False))

plt.figure(figsize=(6, 5))
plt.scatter(yt, pt, alpha=0.12, s=8)
plt.plot([0,4], [0,4], linestyle="--")
plt.xlabel("Nearest-neighbour pseudo-next stress class")
plt.ylabel("Q-learning estimated next stress class")
plt.title("Test pseudo-transition comparison")
plt.tight_layout(); plt.show()

## Step 6 — Separate rule-constrained management recommendation layer

The RL output estimates the next stress class. The recommendation layer then applies the manuscript's Table 4 thresholds independently.

- Soil moisture `< 30%` → irrigation.
- Nitrogen `< 20 mg/kg` → nitrogen fertilisation, **only if a nitrogen column is present**.
- Potassium `< 15 mg/kg` → potassium fertilisation, **only if a potassium column is present**.
- Crop switching → only if severe stress and at least two independently measured stress constraints are available and triggered.
- Otherwise → no action.

The uploaded CSV currently has no nitrogen or potassium concentration columns. Therefore, N/K recommendations will be marked unavailable rather than fabricated. `PH_DEVIATION_THRESHOLD` must be supplied from an agronomic justification before pH can count as a second constraint for crop switching.

In [ ]:
ACTION_MAP = {
    0: "No Action",
    1: "Irrigation",
    2: "Nitrogen Fertilization",
    3: "Potassium Fertilization",
    4: "Crop Switching",
}

N_COL = next((c for c in ["Nitrogen", "Nitrogen_Level", "Nitrogen_Content", "Soil_Nitrogen"] if c in df.columns), None)
K_COL = next((c for c in ["Potassium", "Potassium_Level", "Potassium_Content", "Soil_Potassium"] if c in df.columns), None)


def rule_constrained_recommendation(row, predicted_stress):
    moisture_low = float(row["Soil_Moisture"]) < 30.0
    n_low = (N_COL is not None) and (float(row[N_COL]) < 20.0)
    k_low = (K_COL is not None) and (float(row[K_COL]) < 15.0)
    ph_low, ph_high = PH_OPTIMUM_RANGES[str(row["Crop_Type"])]
    pH_deviation = max(ph_low - float(row["Soil_pH"]), 0.0) + max(float(row["Soil_pH"]) - ph_high, 0.0)
    ph_stress = (PH_DEVIATION_THRESHOLD is not None) and (pH_deviation >= float(PH_DEVIATION_THRESHOLD))

    independent_constraints = int(moisture_low) + int(n_low) + int(k_low) + int(ph_stress)
    if int(predicted_stress) >= 4 and independent_constraints >= 2:
        return ACTION_MAP[4]
    if moisture_low:
        return ACTION_MAP[1]
    if n_low:
        return ACTION_MAP[2]
    if k_low:
        return ACTION_MAP[3]
    return ACTION_MAP[0]


def add_rl_predictions(frame, states):
    out = frame.copy()
    deltas = np.array([predict_increment(Q_table, s) for s in states], dtype=int)
    out["RL_Predicted_Stress_Increment"] = deltas
    out["RL_Predicted_Next_Stress"] = np.minimum(4, out["Stress_Level"].to_numpy(dtype=int) + deltas)
    out["Recommended_Action"] = [rule_constrained_recommendation(row, stress) for (_, row), stress in zip(out.iterrows(), out["RL_Predicted_Next_Stress"])]
    return out

val_results = add_rl_predictions(val_df, val_states)
test_results = add_rl_predictions(test_df, test_states)

print("N concentration column:", N_COL)
print("K concentration column:", K_COL)
print("\nTest recommendation counts:")
print(test_results["Recommended_Action"].value_counts())
print("\nSample test outputs:")
print(test_results[["Stress_Level", "RL_Predicted_Stress_Increment", "RL_Predicted_Next_Stress", "Soil_Moisture", "Soil_pH", "Recommended_Action"]].head(10))

## Step 7 — Transparent behavioural summaries and export

These are descriptive diagnostics, not independent agronomic validation. In particular, agreement between a recommendation and the same rule that generated it must not be called physiological validation. The test outputs and all training-derived parameters are exported for auditability.

In [ ]:
# Action distribution by RL-estimated stress class
action_table = pd.crosstab(test_results["RL_Predicted_Next_Stress"], test_results["Recommended_Action"], normalize="index")
print("Action proportions by predicted stress class:")
print(action_table.round(3))

action_table.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.xlabel("RL-estimated next stress class")
plt.ylabel("Proportion")
plt.title("Rule-constrained recommendations by RL-estimated stress")
plt.legend(title="Recommendation", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

# Save outputs and model configuration.
OUT_DIR = Path("optionA_outputs")
OUT_DIR.mkdir(exist_ok=True)
test_results.to_csv(OUT_DIR / "test_predictions_and_recommendations.csv", index=False)
val_results.to_csv(OUT_DIR / "validation_predictions_and_recommendations.csv", index=False)
metrics_df.to_csv(OUT_DIR / "pseudo_transition_metrics.csv", index=False)

config = {
    "random_state": RANDOM_STATE,
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "csi_params": CSI_PARAMS,
    "csi_cut_points_training_quintiles": CSI_CUTS.tolist(),
    "moisture_direction": MOISTURE_DIRECTION,
    "chlorophyll_direction": CHLOROPHYLL_DIRECTION,
    "crop_specific_ph_optimum_ranges": PH_OPTIMUM_RANGES,
    "ph_deviation_threshold": PH_DEVIATION_THRESHOLD,
    "state_features": STATE_FEATURES,
    "state_bins": {k: v.tolist() for k, v in STATE_BINS.items()},
    "q_learning": {"alpha": ALPHA, "gamma": GAMMA, "epsilon": EPSILON, "episodes": EPISODES, "actions_delta_stress": ACTIONS.tolist()},
    "pseudo_transition_note": "Nearest-neighbour similarity transitions from training data; not chronological observations.",
    "missing_n_k_columns": {"nitrogen": N_COL is None, "potassium": K_COL is None},
    "test_metrics": test_metrics,
}
with open(OUT_DIR / "model_configuration_and_results.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\nSaved outputs in: {OUT_DIR.resolve()}")

## Step 8 — What must be corrected in the manuscript after running

1. Describe the CSI classes as **relative CSI-derived categories**, not independently annotated physiological ground truth.
2. Describe Q-learning as learning a stress-increment policy over **training-only nearest-neighbour pseudo-transitions**; do not call these real temporal observations.
3. Report pseudo-transition MAE separately from any genuine future-forecast metric. The current dataset cannot provide the latter.
4. State that nitrogen/potassium rules were not operational because those concentration columns are absent, unless additional measured data are provided.
5. Recompute every figure/table from the exported outputs. Delete old convergence, MAE, SHAP, and action claims that cannot be regenerated.
6. SHAP must explain a clearly specified supervised model trained against an independently defined target. Do not train a model to reproduce rule-generated labels and present that as independent stress validation.
7. A real temporal validation study requires timestamped repeated observations from the same plots/fields, preferably with independently measured stress outcomes.